# 08 — Full Backtest
## Expanding-Window Validation on 2023–2024 Holdout

**This is the moment of truth.**

The backtest walks through every holdout tournament chronologically, using ONLY data available at the time of each prediction. No future data leakage.

**Three validation gates:**
1. **Calibration:** Model Brier Score < Market Brier Score
2. **Significance:** Diebold-Mariano p-value < 0.05
3. **Betting Viability:** ROI > 0%, Sharpe > 0.5, Max DD < 40%

All three must pass before live deployment.


In [ ]:
# --- Setup ---
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config.settings import Settings
from data.loader import DataLoader
from features.pipeline import FeaturePipeline
from models.baseline import BaselineModel
from validation.backtest import BacktestEngine
from validation.metrics import generate_validation_report
from validation.statistical_tests import diebold_mariano_test, bootstrap_test

cfg = Settings()
cfg.OVERROUND_METHOD = "proportional"  # Power method over-compresses longshot probs, creating phantom edges
loader = DataLoader(cfg)

In [ ]:
# --- Load All Data ---
rounds_df = loader.load_rounds()

# Normalize training rounds: rename dg_id → player_id and create date from event_completed
if "dg_id" in rounds_df.columns and "player_id" not in rounds_df.columns:
    rounds_df = rounds_df.rename(columns={"dg_id": "player_id"})
if "date" not in rounds_df.columns and "event_completed" in rounds_df.columns:
    rounds_df["date"] = pd.to_datetime(rounds_df["event_completed"], errors="coerce")

try:
    events_df = loader.load_events()
except FileNotFoundError:
    # Infer events from rounds
    events_df = rounds_df.groupby("event_id").agg(
        start_date=("date", "min"),
        event_name=("event_name", "first") if "event_name" in rounds_df.columns else ("event_id", "first"),
    ).reset_index()
    events_df["calendar_year"] = pd.to_datetime(events_df["start_date"]).dt.year
    if "season" in rounds_df.columns:
        events_df["season"] = rounds_df.groupby("event_id")["season"].first().values

# Ensure events_df has start_date for the backtest engine to sort/filter by
if "start_date" not in events_df.columns and "calendar_year" in events_df.columns:
    events_df["start_date"] = pd.to_datetime(
        events_df["calendar_year"].astype(int).astype(str) + "-01-01"
    )

try:
    odds_df = loader.load_odds()
except FileNotFoundError:
    odds_df = pd.DataFrame()
    print("⚠️ No odds data — betting metrics will not be available")

print(f"Rounds: {len(rounds_df):,}")
print(f"Events: {len(events_df):,}")
print(f"Odds:   {len(odds_df):,}")


In [ ]:
# --- Model: ProductionEWMAModel (shared with notebook 09 / live deployment) ---
#
# The model code lives in models/production_ewma.py so the backtest and
# live deployment provably run identical logic. All hyperparameters
# (EWMA half-lives, course/recent-form blends, course-fit shrinkage,
# mu_std cap, simulation count) live in config/settings.py and are
# FROZEN pending out-of-sample validation — do not re-tune them against
# this 2023–2024 holdout (see AUDIT.md finding 2.3).

from models.production_ewma import ProductionEWMAModel

model = ProductionEWMAModel(cfg)
fit_and_predict = model.fit_and_predict

In [ ]:
# --- Diagnostic: Parameter summary ---
print("=== Current Parameters (from config/settings.py — FROZEN) ===")
print(f"EWMA half-life: {cfg.EWMA_HALF_LIFE_ROUNDS} rounds / {cfg.EWMA_HALF_LIFE_DAYS} days")
print(f"Recent-form:    {cfg.RECENT_FORM_BLEND*100:.0f}% of last {cfg.RECENT_FORM_ROUNDS} rounds")
print(f"Course SG wt:   {cfg.COURSE_SG_BLEND*100:.0f}% blend (min {cfg.MIN_COURSE_ROUNDS_FOR_WEIGHTS} hist rounds)")
print(f"Course fit τ:   {cfg.COURSE_FIT_SHRINKAGE}")
print(f"t-scale:        applied inside MonteCarloSimulator (ν={cfg.OBSERVATION_DF})")
print(f"mu_std cap:     {cfg.MU_STD_CAP}")
print(f"Simulations:    {cfg.PRODUCTION_N_SIMULATIONS:,}")

# Check course weight coverage
_diag_train = rounds_df[["event_id", "sg_ott", "sg_app", "sg_arg", "sg_putt"]].dropna()
_event_counts = _diag_train.groupby("event_id").size()
_n_events_with_cw = (_event_counts >= cfg.MIN_COURSE_ROUNDS_FOR_WEIGHTS).sum()
print(f"\nEvents with {cfg.MIN_COURSE_ROUNDS_FOR_WEIGHTS}+ rounds for course weights: "
      f"{_n_events_with_cw}/{len(_event_counts)} ({_n_events_with_cw/len(_event_counts)*100:.0f}%)")

# Show example course weights for a few events
print(f"\n=== Example Course SG Weights ===")
for eid in _event_counts.nlargest(3).index:
    _evt = _diag_train[_diag_train["event_id"] == eid][["sg_ott","sg_app","sg_arg","sg_putt"]]
    _var = _evt.var()
    _w = _var / _var.sum()
    print(f"  Event {eid} ({len(_evt)} rounds): "
          f"ott={_w['sg_ott']:.2f} app={_w['sg_app']:.2f} "
          f"arg={_w['sg_arg']:.2f} putt={_w['sg_putt']:.2f}")

del _diag_train, _event_counts, _n_events_with_cw


In [ ]:
# --- Load holdout data from subfolder ---
holdout_dir = cfg.DATA_DIR / "holdout"
rounds_holdout = pd.read_csv(holdout_dir / "sg_rounds_pga_2023_2024.csv", low_memory=False)
rounds_holdout = rounds_holdout.rename(columns={"dg_id": "player_id"})
# Use actual tournament completion date, not a year-level approximation
rounds_holdout["date"] = pd.to_datetime(rounds_holdout["event_completed"], errors="coerce")

odds_holdout = pd.read_csv(holdout_dir / "odds_2023_2024.csv", low_memory=False)

schedule_holdout = pd.read_csv(holdout_dir / "schedule_2023_2024.csv", low_memory=False)
# PGA only — rounds data is PGA-only, other tours would just be skipped anyway
schedule_holdout = schedule_holdout[
    schedule_holdout["calendar_year"].isin([2023, 2024]) &
    (schedule_holdout["tour"] == "pga")
].copy()
schedule_holdout["start_date"] = pd.to_datetime(schedule_holdout["date"], errors="coerce")

# --- Combine events, rounds, odds ---
events_combined = pd.concat([events_df, schedule_holdout], ignore_index=True)
rounds_combined = pd.concat([rounds_df, rounds_holdout], ignore_index=True)
odds_combined   = pd.concat([odds_df,   odds_holdout],   ignore_index=True)

# Rename odds columns to match what process_tournament_odds() expects:
#   dg_id → player_id, bookmaker → book, close_odds → decimal_odds
odds_combined = odds_combined.rename(columns={
    "dg_id": "player_id",
    "bookmaker": "book",
    "close_odds": "decimal_odds",
})
# Filter to win market only (both training and holdout files contain top_5, make_cut, etc.)
if "market" in odds_combined.columns:
    odds_combined = odds_combined[odds_combined["market"].str.lower() == "win"]

print(f"Holdout events (PGA 2023-2024): {len(schedule_holdout):,}")
print(f"Holdout rounds:                 {len(rounds_holdout):,}")
print(f"Combined rounds:                {len(rounds_combined):,}")

# --- One-time prep: parse dates and pre-sort rounds ---
rounds_combined["date"] = pd.to_datetime(rounds_combined["date"])
rounds_combined = rounds_combined.sort_values("date").reset_index(drop=True)
print(f"Rounds pre-sorted by date. Ready for backtest.")

In [ ]:
# --- Run full backtest (SKIP if cache already exists) ---
# This takes ~30-60 min. Only run if you need to regenerate the cache.
# If cache exists, skip to the next cell (Fast Re-Evaluation).

from pathlib import Path

CACHE_PATH = cfg.PROCESSED_DIR / "backtest_cache" / "ewma25_50k_v3.pkl"

engine = BacktestEngine(cfg)
print("Starting expanding-window backtest...")
print(f"Holdout seasons: {cfg.HOLDOUT_SEASONS}")
print(f"Cache will be saved to: {CACHE_PATH}")
print("Estimated time: ~30-60 minutes (50K sims + H2H pairwise)\n")

result = engine.run(
    events_df=events_combined,
    rounds_df=rounds_combined,
    odds_df=odds_combined,
    fit_and_predict_fn=fit_and_predict,
    holdout_seasons=cfg.HOLDOUT_SEASONS,
    cache_path=CACHE_PATH,
    model_description="EWMA 25r/120d + course-fit + recent-form + 50K sims + H2H",
)

print(f"\n{'='*60}")
print(f"BACKTEST COMPLETE")
print(f"{'='*60}")
print(f"Events evaluated:    {result.total_events}")
print(f"Model avg Brier:     {result.model_avg_brier:.5f}")
print(f"Market avg Brier:    {result.market_avg_brier:.5f}")
print(f"ROI:                 {result.roi_pct:.1f}%")
print(f"Sharpe:              {result.sharpe:.2f}")
print(f"Max Drawdown:        {result.max_dd_pct:.1f}%")
print(f"\nCache saved to: {CACHE_PATH}")

In [ ]:
# --- Fast Re-Evaluation (seconds, not minutes) ---
# Load cached model predictions and re-run ONLY the betting pipeline.

# Reload to pick up any code changes without kernel restart
import importlib
import validation.backtest
importlib.reload(validation.backtest)
from validation.backtest import BacktestCache, BacktestEngine

CACHE_PATH = cfg.PROCESSED_DIR / "backtest_cache" / "ewma25_50k_v3.pkl"
cache = BacktestCache.load(CACHE_PATH)
print(f"Loaded cache: {cache.n_events} events, created {cache.created_at}")
print(f"Model: {cache.model_description}")
print(f"H2H data: {'yes' if cache.h2h_predictions else 'no'}")
print(f"Top-5 data: {'yes' if cache.top5_predictions else 'no'}")

# --- Outright winner betting parameters ---
cfg.PROB_TEMPERATURE = 0.35        # Sharpen model probs
cfg.OVERROUND_METHOD = "proportional"
cfg.MIN_BET_PROBABILITY = 0.05     # Only bet where P_market >= 5%
cfg.MIN_EDGE_THRESHOLD = 1.50      # Require 50% edge (high-conviction only)
cfg.KELLY_FRACTION = 0.25

engine = BacktestEngine(cfg)
result = engine.evaluate(cache, events_combined, odds_combined)

print(f"\n{'='*60}")
print(f"OUTRIGHT WINNER RE-EVALUATION")
print(f"{'='*60}")
print(f"Events evaluated:    {result.total_events}")
print(f"Model avg Brier:     {result.model_avg_brier:.5f}")
print(f"Market avg Brier:    {result.market_avg_brier:.5f}")
print(f"ROI:                 {result.roi_pct:.1f}%")
print(f"Sharpe:              {result.sharpe:.2f}")
print(f"Max Drawdown:        {result.max_dd_pct:.1f}%")

In [ ]:
# --- Diagnostic: Bet Quality Analysis ---
# Scrutinize: Is the ROI realistic or driven by a few lucky wins?

import importlib, validation.backtest
importlib.reload(validation.backtest)

from betting.edge_detection import EdgeDetector
from betting.odds_processing import process_tournament_odds

events_with_bets = [e for e in result.events if e.n_bets > 0]
total_bets = sum(e.n_bets for e in result.events)
won_events = [e for e in events_with_bets if e.pnl > 0]

print(f"Events with bets: {len(events_with_bets)} / {result.total_events}")
print(f"Total bets placed: {total_bets}")
print(f"Events with winning bets: {len(won_events)}")
print(f"Win rate (event-level): {len(won_events)/max(len(events_with_bets),1)*100:.1f}%")
print(f"Total staked: ${result.total_staked:.2f}")
print(f"Total P&L: ${result.total_pnl:.2f}")

# --- Check model flatness vs market ---
model_stds, market_stds = [], []
model_maxes, market_maxes = [], []
for e in result.events:
    if e.model_probs and e.market_probs:
        common = set(e.model_probs) & set(e.market_probs)
        if len(common) < 5:
            continue
        mp = np.array([e.model_probs[p] for p in common])
        mkp = np.array([e.market_probs[p] for p in common])
        model_stds.append(mp.std())
        market_stds.append(mkp.std())
        model_maxes.append(mp.max())
        market_maxes.append(mkp.max())

print(f"\n{'='*50}")
print(f"PROBABILITY DISTRIBUTION SHAPE")
print(f"{'='*50}")
print(f"Model  std (avg): {np.mean(model_stds):.5f}")
print(f"Market std (avg): {np.mean(market_stds):.5f}")
print(f"Ratio model/market: {np.mean(model_stds)/np.mean(market_stds):.2f}x")
print(f"Model  max P(win) avg: {np.mean(model_maxes):.4f}")
print(f"Market max P(win) avg: {np.mean(market_maxes):.4f}")

# --- Bet characteristics ---
print(f"\n{'='*50}")
print(f"BET CHARACTERISTICS")
print(f"{'='*50}")
bet_odds_list = []
bet_pmodel_list = []
bet_pmarket_list = []
bet_edge_list = []
for e in events_with_bets:
    if not e.model_probs or not e.market_probs:
        continue
    for pid in e.model_probs:
        if pid not in e.market_probs:
            continue
        pm = e.model_probs[pid]
        pmk = e.market_probs[pid]
        if pmk >= cfg.MIN_BET_PROBABILITY and pmk > 0 and pm / pmk > cfg.MIN_EDGE_THRESHOLD:
            bet_pmodel_list.append(pm)
            bet_pmarket_list.append(pmk)
            bet_edge_list.append(pm / pmk - 1)
            bet_odds_list.append(1.0 / pmk if pmk > 0 else 0)

if bet_odds_list:
    print(f"Avg model P(win):   {np.mean(bet_pmodel_list):.4f} ({1/np.mean(bet_pmodel_list):.0f}:1)")
    print(f"Avg market P(win):  {np.mean(bet_pmarket_list):.4f} ({1/np.mean(bet_pmarket_list):.0f}:1)")
    print(f"Avg implied odds:   {np.mean(bet_odds_list):.1f}")
    print(f"Avg edge:           {np.mean(bet_edge_list)*100:.1f}%")
    longshot_frac = np.mean(np.array(bet_pmarket_list) < 0.03)
    print(f"Bets on longshots (P_market < 3%): {longshot_frac*100:.0f}%")

# --- CRITICAL: Concentration risk ---
print(f"\n{'='*50}")
print(f"CONCENTRATION RISK (is ROI driven by 1-2 lucky wins?)")
print(f"{'='*50}")
pnl_by_event = [(e.event_name, e.pnl, e.n_bets, e.stake_total) for e in result.events if e.n_bets > 0]
pnl_by_event.sort(key=lambda x: x[1], reverse=True)

total_pnl = sum(x[1] for x in pnl_by_event)
print(f"\nTop 5 events by P&L:")
cumulative = 0
for name, pnl, n, stake in pnl_by_event[:5]:
    cumulative += pnl
    pct_of_total = (pnl / total_pnl * 100) if total_pnl != 0 else 0
    print(f"  {name[:35]:35s}  P&L=${pnl:>8.2f}  ({pct_of_total:>5.1f}% of total)  bets={n}")
print(f"  Top 5 cumulative: ${cumulative:.2f} ({cumulative/total_pnl*100:.0f}% of total P&L)" if total_pnl != 0 else "")

print(f"\nBottom 5 events by P&L:")
for name, pnl, n, stake in pnl_by_event[-5:]:
    pct_of_total = (pnl / total_pnl * 100) if total_pnl != 0 else 0
    print(f"  {name[:35]:35s}  P&L=${pnl:>8.2f}  ({pct_of_total:>5.1f}% of total)  bets={n}")

# How many winning events does it take to explain all the P&L?
if total_pnl > 0:
    sorted_pnl = sorted([x[1] for x in pnl_by_event], reverse=True)
    cumsum = np.cumsum(sorted_pnl)
    n_events_for_all_pnl = np.searchsorted(cumsum, total_pnl) + 1
    print(f"\n  {n_events_for_all_pnl} of {len(pnl_by_event)} events account for 100% of P&L")

In [ ]:
# --- H2H Matchup Backtest ---
# The primary betting market: head-to-head matchups (~80-100+ per event).

import importlib
import validation.backtest
importlib.reload(validation.backtest)
from validation.backtest import BacktestCache, BacktestEngine

# Load matchup odds (holdout 2023-2024)
matchup_odds = pd.read_csv(cfg.DATA_DIR / "holdout" / "matchup_odds_2023_2024.csv")
print(f"Matchup odds loaded: {len(matchup_odds)} rows, {matchup_odds['event_id'].nunique()} events")
print(f"Bet types: {matchup_odds['bet_type'].value_counts().to_dict()}")

# Load cache
CACHE_PATH = cfg.PROCESSED_DIR / "backtest_cache" / "ewma25_50k_v3.pkl"
cache = BacktestCache.load(CACHE_PATH)

if not cache.h2h_predictions:
    print("\nCache has no H2H predictions. Re-run full backtest with compute_h2h=True.")
else:
    print(f"\nCache has H2H data for {len(cache.h2h_predictions)} events")

    # --- Matchup betting parameters ---
    cfg.MATCHUP_MIN_EDGE = 0.08       # 8% probability edge required (sweet spot from grid search)
    cfg.KELLY_FRACTION = 0.25
    cfg.INITIAL_BANKROLL = 5000.0

    engine = BacktestEngine(cfg)
    h2h_result = engine.evaluate_matchups(
        cache, matchup_odds, bet_type="72-hole Match"
    )

    print(f"\n{'='*60}")
    print(f"H2H MATCHUP BACKTEST RESULTS")
    print(f"{'='*60}")
    print(f"Total bets:         {h2h_result['n_bets']}")
    print(f"Win rate:           {h2h_result['win_rate']:.1f}%")
    print(f"Total staked:       ${h2h_result['total_staked']:,.2f}")
    print(f"Total P&L:          ${h2h_result['total_pnl']:,.2f}")
    print(f"ROI:                {h2h_result['roi_pct']:.1f}%")
    print(f"Sharpe:             {h2h_result['sharpe']:.2f}")
    print(f"Max Drawdown:       {h2h_result['max_dd_pct']:.1f}%")
    print(f"Final Bankroll:     ${h2h_result['final_bankroll']:,.2f}")
    print(f"Events with bets:   {h2h_result['n_events_with_bets']}")

    # Sanity check: bet-level breakdown
    bets_df = h2h_result['bets_df']
    if len(bets_df) > 0:
        print(f"\n--- Bet-level stats ---")
        print(f"Avg edge:           {bets_df['edge'].mean()*100:.1f}%")
        print(f"Avg decimal odds:   {bets_df['decimal_odds'].mean():.2f}")
        print(f"Avg stake:          ${bets_df['stake'].mean():.2f}")
        print(f"Median stake:       ${bets_df['stake'].median():.2f}")

        # Per-event breakdown
        event_summary = bets_df.groupby("event_name").agg(
            n_bets=("pnl", "count"),
            pnl=("pnl", "sum"),
            staked=("stake", "sum"),
            win_rate=("won", "mean"),
        ).sort_values("pnl", ascending=False)
        event_summary["roi_pct"] = (event_summary["pnl"] / event_summary["staked"] * 100).round(1)
        print(f"\n--- Top 5 events by P&L ---")
        for name, row in event_summary.head(5).iterrows():
            print(f"  {name[:35]:35s}  bets={int(row['n_bets']):3d}  "
                  f"P&L=${row['pnl']:>8.2f}  ROI={row['roi_pct']:>5.1f}%  WR={row['win_rate']*100:.0f}%")
        print(f"\n--- Bottom 5 events by P&L ---")
        for name, row in event_summary.tail(5).iterrows():
            print(f"  {name[:35]:35s}  bets={int(row['n_bets']):3d}  "
                  f"P&L=${row['pnl']:>8.2f}  ROI={row['roi_pct']:>5.1f}%  WR={row['win_rate']*100:.0f}%")

In [ ]:
# --- H2H Matchup Edge Grid Search ---
# Sweep MATCHUP_MIN_EDGE to find the optimal threshold.

import importlib, validation.backtest
importlib.reload(validation.backtest)
from validation.backtest import BacktestCache, BacktestEngine

CACHE_PATH = cfg.PROCESSED_DIR / "backtest_cache" / "ewma25_50k_v3.pkl"
cache = BacktestCache.load(CACHE_PATH)
matchup_odds = pd.read_csv(cfg.DATA_DIR / "holdout" / "matchup_odds_2023_2024.csv")

print(f"{'Edge':>6} | {'Bets':>5} | {'WinRate':>7} | {'ROI%':>7} | {'Sharpe':>6} | {'MaxDD%':>6} | {'P&L':>10}")
print("-" * 65)

for edge in [0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.10, 0.12, 0.15]:
    cfg.MATCHUP_MIN_EDGE = edge
    cfg.KELLY_FRACTION = 0.25
    cfg.INITIAL_BANKROLL = 5000.0
    engine = BacktestEngine(cfg)
    r = engine.evaluate_matchups(cache, matchup_odds, bet_type="72-hole Match")
    print(f"{edge:>6.2f} | {r['n_bets']:>5} | {r['win_rate']:>6.1f}% | {r['roi_pct']:>6.1f}% | "
          f"{r['sharpe']:>6.2f} | {r['max_dd_pct']:>5.1f}% | ${r['total_pnl']:>9.2f}")

In [ ]:
# --- Flat vs Edge-Scaled Bet Sizing Comparison ---
# Compare the old flat-$25 sizing (cap=0.005) vs new Kelly-scaled sizing (cap=0.015).

import importlib, validation.backtest
importlib.reload(validation.backtest)
from validation.backtest import BacktestCache, BacktestEngine

CACHE_PATH = cfg.PROCESSED_DIR / "backtest_cache" / "ewma25_50k_v3.pkl"
cache = BacktestCache.load(CACHE_PATH)
matchup_odds = pd.read_csv(cfg.DATA_DIR / "holdout" / "matchup_odds_2023_2024.csv")

cfg.MATCHUP_MIN_EDGE = 0.08
cfg.KELLY_FRACTION = 0.25
cfg.INITIAL_BANKROLL = 5000.0

results = {}
for label, cap in [("Flat $25 (cap=0.5%)", 0.005), ("Kelly-scaled (cap=1.5%)", 0.015)]:
    cfg.MATCHUP_MAX_BET_PCT = cap
    engine = BacktestEngine(cfg)
    r = engine.evaluate_matchups(cache, matchup_odds, bet_type="72-hole Match")
    bdf = r["bets_df"]
    results[label] = {
        "Bets": r["n_bets"],
        "Win Rate": f"{r['win_rate']:.1f}%",
        "ROI": f"{r['roi_pct']:.1f}%",
        "Sharpe": f"{r['sharpe']:.2f}",
        "Max DD": f"{r['max_dd_pct']:.1f}%",
        "Total P&L": f"${r['total_pnl']:,.2f}",
        "Final Bankroll": f"${r['final_bankroll']:,.2f}",
        "Avg Stake": f"${bdf['stake'].mean():.2f}" if len(bdf) > 0 else "$0",
        "Stake Std": f"${bdf['stake'].std():.2f}" if len(bdf) > 0 else "$0",
        "Stake Range": f"${bdf['stake'].min():.0f}–${bdf['stake'].max():.0f}" if len(bdf) > 0 else "N/A",
    }

# Restore the new setting
cfg.MATCHUP_MAX_BET_PCT = 0.015

# Display comparison
comp_df = pd.DataFrame(results)
print("=" * 70)
print("FLAT vs KELLY-SCALED BET SIZING — H2H MATCHUP BACKTEST (2023-2024)")
print("=" * 70)
print(comp_df.to_string())
print("\nVerdict: Kelly-scaled is better if ROI and Sharpe are equal or higher.")


In [ ]:
# --- Gate Verdicts ---
import importlib, validation.backtest
importlib.reload(validation.backtest)

print(f"{'='*60}")
print(f"VALIDATION GATES — OUTRIGHT WINNER")
print(f"{'='*60}")
print(f"Gate 1 (Calibration):  {'PASS' if result.gate_1_passed else 'FAIL'}")
print(f"  Model Brier < Market: {result.model_avg_brier:.5f} {'<' if result.gate_1_passed else '>'} {result.market_avg_brier:.5f}")
print()
print(f"Gate 3 (Betting):      {'PASS' if result.gate_3_passed else 'FAIL'}")
print(f"  ROI > 0%:      {result.roi_pct:.1f}%")
print(f"  Sharpe > 0.5:  {result.sharpe:.2f}")
print(f"  Max DD < 40%:  {result.max_dd_pct:.1f}%")

# Matchup gates (if h2h_result exists)
try:
    print(f"\n{'='*60}")
    print(f"VALIDATION GATES — H2H MATCHUPS")
    print(f"{'='*60}")
    h2h_roi_pass = h2h_result['roi_pct'] > 0
    h2h_sharpe_pass = h2h_result['sharpe'] > 0.3
    h2h_sample_pass = h2h_result['n_bets'] >= 100
    h2h_wr_pass = h2h_result['win_rate'] > 50.0

    print(f"Gate M1 (Sample size):  {'PASS' if h2h_sample_pass else 'FAIL'}  ({h2h_result['n_bets']} bets, need >= 100)")
    print(f"Gate M2 (Win rate):     {'PASS' if h2h_wr_pass else 'FAIL'}  ({h2h_result['win_rate']:.1f}%, need > 50%)")
    print(f"Gate M3 (ROI > 0):      {'PASS' if h2h_roi_pass else 'FAIL'}  ({h2h_result['roi_pct']:.1f}%)")
    print(f"Gate M4 (Sharpe > 0.3): {'PASS' if h2h_sharpe_pass else 'FAIL'}  ({h2h_result['sharpe']:.2f})")
    print(f"Max Drawdown:           {h2h_result['max_dd_pct']:.1f}%")

    all_matchup_gates = h2h_roi_pass and h2h_sharpe_pass and h2h_sample_pass and h2h_wr_pass
    print(f"\nMatchup gates: {'ALL PASSED' if all_matchup_gates else 'SOME FAILED'}")
except NameError:
    print("\n(H2H matchup results not available — run matchup backtest cell first)")

In [ ]:
# --- Diebold-Mariano Test ---
if result.total_events > 10 and result.market_avg_brier > 0:
    model_briers = np.array([e.model_brier for e in result.events])
    market_briers = np.array([e.market_brier for e in result.events if e.market_brier > 0])
    
    # Align arrays (only events with both model and market scores)
    aligned_events = [e for e in result.events if e.market_brier > 0]
    m_b = np.array([e.model_brier for e in aligned_events])
    mk_b = np.array([e.market_brier for e in aligned_events])
    
    if len(m_b) >= 10:
        dm_result = diebold_mariano_test(m_b, mk_b)
        print(f"\nDiebold-Mariano Test:")
        print(f"  DM statistic: {dm_result['dm_statistic']:.3f}")
        print(f"  p-value:      {dm_result['p_value']:.4f}")
        print(f"  Significant:  {'✅ YES' if dm_result['significant_005'] else '❌ NO'}")
        print(f"  Model better in {dm_result['model_better_pct']:.0f}% of events")
        
        boot = bootstrap_test(m_b, mk_b)
        print(f"\nBootstrap Test:")
        print(f"  95% CI: [{boot['ci_95_lower']:.6f}, {boot['ci_95_upper']:.6f}]")
        print(f"  Significant: {'✅ YES' if boot['significant'] else '❌ NO'}")
else:
    print("Insufficient data for statistical significance testing")


In [ ]:
# --- Visualization ---
from utils.plotting import plot_brier_comparison, plot_backtest_cumulative_pnl

per_event_df = engine.results_to_dataframe(result)

if len(per_event_df) > 0:
    # Brier comparison
    m_b = per_event_df["model_brier"].values
    mk_b = per_event_df["market_brier"].values
    if mk_b.sum() > 0:
        fig, ax = plot_brier_comparison(m_b, mk_b)
        plt.show()

    # Cumulative P&L (if betting data available)
    if per_event_df["pnl"].sum() != 0:
        fig, ax = plot_backtest_cumulative_pnl(per_event_df["pnl"].values)
        plt.show()
        
print("\nPer-event results:")
print(per_event_df.head(20).to_string(index=False))


---
## ✅ Backtest Complete

**If all gates passed:** → Proceed to **09_live_deployment.ipynb**

**If gates failed (expected first time!):**
1. Check calibration curves — is the model over/under-confident?
2. Try different EWMA half-life values
3. Switch from baseline to Bayesian model (Notebook 04)
4. Add course-fit if not already included
5. Check for data quality issues

**Remember:** The edge in golf betting is small. A model that fails now may succeed after refinement. The validation framework ensures you don't deploy prematurely.
